In [1]:
import pandas as pd

In [2]:
df_injured = pd.read_csv("accidentes-injury-histo.csv")

/tmp/ipykernel_34874/2128208587.py:1: DtypeWarning: Columns (0: case_id_pkey, 1: juris) have mixed types. Specify dtype option on import or set low_memory=False.
  df_injured = pd.read_csv("accidentes-injury-histo.csv")


In [3]:
df_injured.info()

<class 'pandas.DataFrame'>
RangeIndex: 64694 entries, 0 to 64693
Data columns (total 58 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   unique_id                64694 non-null  int64  
 1   cnn_intrsctn_fkey        64657 non-null  float64
 2   cnn_sgmt_fkey            28914 non-null  float64
 3   case_id_pkey             64694 non-null  object 
 4   tb_latitude              64513 non-null  str    
 5   tb_longitude             64513 non-null  str    
 6   geocode_source           64694 non-null  str    
 7   geocode_location         64694 non-null  str    
 8   collision_datetime       64694 non-null  str    
 9   collision_date           64694 non-null  str    
 10  collision_time           64632 non-null  str    
 11  accident_year            64694 non-null  int64  
 12  month                    64694 non-null  str    
 13  day_of_week              64685 non-null  str    
 14  time_cat                 64641 no

### 1. Columnas de localización.
1. Point es identico a tb_latitude y tb_longitude. *Sería la columna a usar en una visualización en mapa.*
2. geocode_source y geocode_location son irrelevantes, la primeraespecifica qué agente es al fuente geográfico, la segunda tiene un único valor.
3. El resto de coumnas son relevantes y nos hablan tanto de en qué calle a ocurrido el accidente, cómo poder dibujar la dirección o el final del accidente si fue mi grabe. Puede ser interesante la columna de "distancia recorrida".

In [15]:
geo_c = ['tb_latitude', 'tb_longitude', 'geocode_source', 'geocode_location', 'juris', 'primary_rd', 'secondary_rd', 'distance', 'direction', 'intersection', 'street_view', 'point']
df_injured.loc[:, geo_c].sample(3)

,tb_latitude,tb_longitude,geocode_source,geocode_location,juris,primary_rd,secondary_rd,distance,direction,intersection,street_view,point
6028,"37,78118954126","-122,46112396893",SFPD-CROSSROADS,CITY STREET,3801,GEARY BLVD,03RD AVE,25.0,West,Intersection Rear End <= 150ft,https://maps.google.com/maps?q=&layer=c&cbll=3...,POINT (-122.461123969 37.781189541)
21184,"37,73157607051","-122,49905897633",SFPD-CROSSROADS,CITY STREET,3801,SKYLINE BLVD,ZOO RD,0.0,Not Stated,Intersection <= 20ft,https://maps.google.com/maps?q=&layer=c&cbll=3...,POINT (-122.499058976 37.731576071)
44436,"37,80489601043","-122,41176648127",SFPD-CROSSROADS,CITY STREET,3801,FRANCISCO ST,POWELL ST,0.0,Not Stated,Intersection <= 20ft,https://maps.google.com/maps?q=&layer=c&cbll=3...,POINT (-122.411766481 37.80489601)


**geocode_location** *Eliminar*, unicamente hay una categoría.

In [6]:
df_injured.geocode_location.unique()

<StringArray>
['CITY STREET']
Length: 1, dtype: str

### 2. Columnas de temporales.
1. Esta tabla nos da mucha granularidad a la hora de poder mostrar el momento del accidente. Para el modelo no será útil pero para el BI sí.

In [19]:
time_c = ['collision_datetime', 'collision_date', 'collision_time', 'accident_year', 'month', 'day_of_week', 'time_cat']
df_injured.loc[:, time_c].sample(3)

,collision_datetime,collision_date,collision_time,accident_year,month,day_of_week,time_cat
23255,2014 Jun 28 09:00:00 PM,2014 June 28,21:00:00,2014,June,Saturday,6:01 pm to 10:00 pm
64318,2019 Jun 09 03:46:00 PM,2019 June 09,15:46:00,2019,June,Sunday,2:01 pm to 6:00 pm
3148,2021 May 04 10:45:00 AM,2021 May 04,10:45:00,2021,May,Tuesday,10:01 am to 2:00 pm


### 3. Columnas de policía.
1. Información sobre los ajentes que reportan y sus comisarias.
2. **Necesario decidir si se mostrarán en el negocio o no (importante para un ayuntamiento??)**

In [20]:
police_c = ['officer_id', 'reporting_district', 'beat_number', 'supervisor_district', 'police_district', 'control_device']
df_injured.loc[:, police_c].sample(3)

,officer_id,reporting_district,beat_number,supervisor_district,police_district,control_device
44192,2179,NaN,3A60,3.0,CENTRAL,Functioning
35308,327,Mission,3D11C,8.0,MISSION,NaN
29758,1584,SOUTH,4B5F,6.0,SOUTHERN,Functioning


### 4. Columnas de colisión.
1. Aportan gran cantidad de información sobre el sentido del accidente. Naturaleza de este, causas y partes implicadas.
2. Columnas muy importantes para un modelo de sinisestralidad. Columnas importantísimas cómo: *mviw* y *ped_action*.

In [21]:
colision_c = ['collision_severity', 'type_of_collision', 'mviw', 'ped_action', 'number_killed', 'number_injured']
df_injured.loc[:, colision_c].sample(3)

,collision_severity,type_of_collision,mviw,ped_action,number_killed,number_injured
43510,Injury (Complaint of Pain),Other,Other Object,No Pedestrian Involved,0.0,1
44972,Injury (Complaint of Pain),Vehicle/Pedestrian,Pedestrian,Crossing Not in Crosswalk,0.0,1
30661,Injury (Complaint of Pain),Rear End,Other Motor Vehicle,No Pedestrian Involved,0.0,1


In [10]:
df_injured.collision_severity.value_counts()

collision_severity
Injury (Complaint of Pain)    41129
Injury (Other Visible)        18388
Injury (Severe)                4561
Fatal                           615
Medical                           1
Name: count, dtype: int64

In [11]:
df_injured.type_of_collision.value_counts()

type_of_collision
Broadside             19832
Vehicle/Pedestrian    13389
Rear End              10463
Sideswipe              8553
Head-On                3942
Other                  3391
Hit Object             2465
Not Stated             1507
Overturned             1152
Name: count, dtype: int64

In [12]:
df_injured.mviw.value_counts()

mviw
Other Motor Vehicle               29666
Pedestrian                        15076
Bicycle                            8752
Fixed Object                       2895
Parked Motor Vehicle               2806
Non-Collision                      1531
Not Stated                         1510
Other Object                       1309
Motor Vehicle on Other Roadway      973
Train                               147
Animal                               29
Name: count, dtype: int64

In [13]:
df_injured.ped_action.value_counts()

ped_action
No Pedestrian Involved                       48373
Crossing in Crosswalk at Intersection         9263
Crossing Not in Crosswalk                     3242
In Road, Including Shoulder                   1982
Not in Road                                    869
Not Stated                                     718
Crossing in Crosswalk Not at Intersection      222
Approaching/Leaving School Bus                  14
Not In Road                                     11
Name: count, dtype: int64

### 5. Columnas de carretera.
1. Columans realacionas con el entorno del accidente. **Relación con PCI ??**
1. weather_2 es redundante respecto al 1, no aporta mucho matiz. Al igual que el tiempo, reoad_cond_2 es redundante.

In [23]:
road_c = ['road_surface', 'road_cond_1', 'lighting', 'weather_1', 'weather_2']
df_injured.loc[:, road_c].sample(3)

,road_surface,road_cond_1,lighting,weather_1,weather_2
21893,Dry,No Unusual Condition,Daylight,Clear,Not Stated
30386,Dry,No Unusual Condition,Daylight,Clear,Not Stated
1748,Dry,No Unusual Condition,Daylight,Clear,Not Stated


### 6. Columnas de legalidad.
1. Columans relacionadas sobre qué infracción se a cometido y quien tine la culpa en el parte del seguro **(presupongo ??)**
2. Puede aportar información al modelo, sobre todo la infracción y el tipo de vehículo implicado. **Información repetida respecto a las columnas del accidente.**
3. Sobra muchas columnas. Columnas interesantes: *vz_pcf_description*, *party1_move_pre_acc* y quizas *dph_col_grp_description*

In [24]:
law_c = ['vz_pcf_code', 'vz_pcf_group', 'vz_pcf_description', 'vz_pcf_link', 'dph_col_grp', 'dph_col_grp_description', 'party_at_fault', 'party1_type', 'party1_dir_of_travel', 'party1_move_pre_acc', 'party2_type', 'party2_dir_of_travel', 'party2_move_pre_acc']
df_injured.loc[:, law_c].sample(3)

,vz_pcf_code,vz_pcf_group,vz_pcf_description,vz_pcf_link,dph_col_grp,dph_col_grp_description,party_at_fault,party1_type,party1_dir_of_travel,party1_move_pre_acc,party2_type,party2_dir_of_travel,party2_move_pre_acc
9342,22350,22350,Unsafe speed for prevailing conditions,http://leginfo.legislature.ca.gov/faces/codes_...,AA,Vehicle(s) Only Involved,1.0,Driver,North,Proceeding Straight,Driver,South,Making Left Turn
23121,22107,22107,Unsafe turn or lane change prohibited,http://leginfo.legislature.ca.gov/faces/codes_...,CC,Vehicle-Bicycle,1.0,Driver,East,Making Right Turn,Bicyclist,East,Proceeding Straight
13678,22350,22350,Unsafe speed for prevailing conditions,http://leginfo.legislature.ca.gov/faces/codes_...,AA,Vehicle(s) Only Involved,1.0,Driver,North,Proceeding Straight,NaN,NaN,NaN


In [27]:
df_injured.party1_type.value_counts()

party1_type
Driver            52826
Bicyclist          5234
Pedestrian         5022
Other              1190
Parked Vehicle      381
Not Stated           27
Bicycle               1
Name: count, dtype: int64

In [28]:
df_injured.party1_move_pre_acc.value_counts()

party1_move_pre_acc
Proceeding Straight                       33543
Making Left Turn                          11275
Making Right Turn                          4083
Changing Lanes                             2212
Other                                      1918
Entering Traffic                           1722
Not Stated                                 1388
Backing                                    1384
Stopped In Road                            1210
Making U Turn                              1202
Parked                                      780
Slowing/Stopping                            698
Passing Other Vehicle                       625
Stopped                                     551
Traveling Wrong Way                         488
Ran Off Road                                423
Parking Maneuver                            359
Other Unsafe Turning                        336
Crossed Into Opposing Lane                  226
Merging                                     182
Crossed Into Opposin

In [29]:
df_injured.dph_col_grp_description.value_counts()

dph_col_grp_description
Vehicle(s) Only Involved                    38291
Vehicle-Pedestrian                          15512
Vehicle-Bicycle                              8655
Bicycle Only                                 1188
Bicycle-Pedestrian                            512
Bicycle-Parked Car                            456
Pedestrian Only or Pedestrian-Parked Car       34
Unknown/Not Stated                             19
Vehicle-Bicycle-Pedestrian                     19
Bicycle-Unknown/Not Stated                      6
Name: count, dtype: int64

**Finalmente quedan unas columnas sobre las actualizaciones del dataset que no importan para nuestro problema.**

In [17]:
metadata_c = ['data_as_of', 'data_updated_at', 'data_loaded_at']
df_injured.loc[:, metadata_c].sample(3)

,data_as_of,data_updated_at,data_loaded_at
60176,2019 Jun 20 12:00:00 AM,2025 Apr 28 12:00:00 AM,2026 May 01 12:27:24 PM
47718,2015 Aug 22 12:00:00 AM,2023 Apr 26 12:00:00 AM,2026 May 01 12:27:24 PM
18043,2020 Sep 28 12:00:00 AM,2026 Jan 28 12:00:00 AM,2026 May 01 12:27:24 PM
